In [ ]:
from pathlib import Path
import sys
import psutil
import os

repo_root = Path.cwd()

while not (repo_root / "src").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not find 'src' directory in any parent")
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

p = psutil.Process(os.getpid())
if psutil.WINDOWS:
    p.nice(psutil.HIGH_PRIORITY_CLASS)
else:
    try:
        p.nice(-10)
    except psutil.AccessDenied:
        print("Elevation (sudo) is required to set high priority on Unix.")

In [ ]:
import numpy as np
import json

# Synthetic timing estimator
MAX_BATCH_SIZE = 16
MAX_TOKEN_COUNT = 1000

MEMORY_OVERHEAD_NS = 20_000_000
COMPUTE_UNIT = 200

def expected_model_time_ns(batch_size: int, token_count: int) -> int:
    """Simple monotonic latency estimator in milliseconds."""
    return MEMORY_OVERHEAD_NS + max(10_000_000, batch_size * (token_count ** 2) * COMPUTE_UNIT)

estimators = np.array(
    [
        [expected_model_time_ns(batch_size, token_count) for token_count in range(MAX_TOKEN_COUNT + 1)]
        for batch_size in range(1, MAX_BATCH_SIZE + 1)
    ],
    dtype=np.int64,
)


with open("estimator_arrays/est_store_synthetic.txt", "w") as f:
    json.dump(estimators.tolist(), f, indent=2)

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from demos.real.razors_edge_gpu_benchmark_task import RazorsEdgeGPUBenchmarkTask

initialized_task_gpu = RazorsEdgeGPUBenchmarkTask(ThreadPoolExecutor(1))

with open("estimator_arrays/est_store_bge_m3.txt", "w") as f:
    json.dump(initialized_task_gpu.batch_timing_estimators.tolist(), f, indent=2)



In [ ]:
from concurrent.futures import ThreadPoolExecutor
from demos.cpu.razors_edge_cpu_benchmark_task import RazorsEdgeCPUBenchmarkTask

initialized_task_cpu = RazorsEdgeCPUBenchmarkTask(ThreadPoolExecutor(1))

with open("estimator_arrays/est_store_jina.txt", "w") as f:
    json.dump(initialized_task_cpu.batch_timing_estimators.tolist(), f, indent=2)
